In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
#import decoupler as dc
#from pypath.inputs import msigdb
import seaborn as sns

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from matplotlib.backends.backend_pdf import PdfPages

#import yaml
from pathlib import Path

# for GO enrichment analysis
import tempfile
import tqdm.auto as tqdm
import gseapy as gp
from gseapy.plot import barplot, dotplot
from gseapy import get_library_name
from gseapy import gseaplot
import gseapy.plot as gsplot

#import requests
#import json
#import io

import itertools

#Vulcano plot
#from bioinfokit import analys, visuz

# Output Files

In [ ]:
pdf_dotplots = PdfPages('dotplots.pdf')
pdf_pages= PdfPages('pages.pdf')
output_dir = "/data/projects/manuela/aging/scRNA_aging-mouse/deg_tables_vsSG18/"

# Data import and preparation

In [ ]:
#adata = sc.read("/data/projects/manuela/aging/scRNA_aging-mouse/annotated-aging-soupxed.h5ad", var_names="gene_symbols")
adata = sc.read("/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/annotated-aging-soupxedQpctl.h5ad", var_names="gene_symbols")

adata_PT1 = adata[adata.obs['celltype'] == 'PT-1'].copy()
adata_PT2 = adata[adata.obs['celltype'] == 'PT-2'].copy()
adata_EC2 = adata[adata.obs['celltype'] == 'EC-2'].copy()
adata_TAL1 = adata[adata.obs['celltype'] == 'TAL-1'].copy()
adata_IMM = adata[adata.obs['celltype'] == 'IMM'].copy()

# Gloms
adata_PODO = adata[adata.obs['celltype'] == 'PODO'].copy()
adata_vSMC = adata[adata.obs['celltype'] == 'vSMC/MC'].copy()
adata_PEC = adata[adata.obs['celltype'] == 'PECs?'].copy()
adata_EC1 = adata[adata.obs['celltype'] == 'EC-1(gEC)'].copy()

In [ ]:
sc.get.obs_df(adata, keys=["sample", "n_genes"]
)

In [ ]:
gene_name = 'Ercc'
gene_present = gene_name in adata.var.index
gene_present

# Analysis by Samples by Celltypes

In [ ]:
adata.uns['log1p']['base'] = None
adata_PT1.uns['log1p']['base'] = None
adata_PT2.uns['log1p']['base'] = None
adata_EC2.uns['log1p']['base'] = None
adata_TAL1.uns['log1p']['base'] = None
adata_IMM.uns['log1p']['base'] = None

adata_PODO.uns['log1p']['base'] = None
adata_vSMC.uns['log1p']['base'] = None
adata_PEC.uns['log1p']['base'] = None
adata_EC1.uns['log1p']['base'] = None

sc.tl.rank_genes_groups(adata, 'sample', method='t-test', reference = 'ctrl')
sc.tl.rank_genes_groups(adata_PT1, 'sample', method='t-test', reference = 'ctrl')
sc.tl.rank_genes_groups(adata_PT2, 'sample', method='t-test', reference = 'ctrl')
sc.tl.rank_genes_groups(adata_EC2, 'sample', method='t-test', reference = 'ctrl')
sc.tl.rank_genes_groups(adata_TAL1, 'sample', method='t-test', reference = 'ctrl')
sc.tl.rank_genes_groups(adata_IMM, 'sample', method='t-test', reference = 'ctrl')

# Gloms
sc.tl.rank_genes_groups(adata_PODO, 'sample', method='t-test', reference = 'ctrl')
sc.tl.rank_genes_groups(adata_vSMC, 'sample', method='t-test', reference = 'ctrl')
#sc.tl.rank_genes_groups(adata_PODO, 'sample', method='wilcoxon', reference = 'ctrl')
#sc.tl.rank_genes_groups(adata_vSMC, 'sample', method='wilcoxon', reference = 'ctrl')
sc.tl.rank_genes_groups(adata_PEC, 'sample', method='t-test', reference = 'ctrl')
sc.tl.rank_genes_groups(adata_EC1, 'sample', method='t-test', reference = 'ctrl')

In [ ]:
deg = sc.get.rank_genes_groups_df(adata, group="age")


## DGE Analysis

### pseudo bulk

In [ ]:
p = 0.05
log2fc = 0 

deg = sc.get.rank_genes_groups_df(adata, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg.loc[(deg.logfoldchanges > 0) & (deg.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg.loc[(deg.logfoldchanges < 0) & (deg.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_fi = deg[deg['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_sig = deg[(deg['pvals_adj'] < p) & (deg['logfoldchanges'].abs() > log2fc)]
# Upregulated genes
deg_sig_up = deg_sig[deg_sig['logfoldchanges'] > 0]
# Downregulated genes
deg_sig_dwn = deg_sig[deg_sig['logfoldchanges'] < 0]

deg_sig_sg20 = deg_sig[deg_sig['group']=='age']
deg_sig_up_sg20 = deg_sig_up[deg_sig_up['group']=='age']
deg_sig_dwn_sg20 = deg_sig_dwn[deg_sig_dwn['group']=='age']


deg_sig_sg24 = deg_sig[deg_sig['group']=='sAct']
deg_sig_sg26 = deg_sig[deg_sig['group']=='DR']
deg_sig_sg28 = deg_sig[deg_sig['group']=='combi']

print(deg.shape)
print(deg_fi.shape)
print(deg_sig.shape)

In [ ]:
# Define the list of groups you want to create plots for
groups_to_plot = ['age', 'sAct', 'DR', 'combi']

# Create a column to classify genes as upregulated or downregulated
deg['Regulation'] = 'Not Significant'  # Initialize as not significant
for group in groups_to_plot:
    condition = (deg['pvals_adj'] < 0.05) & (deg['group'] == group)
    deg.loc[condition, 'Regulation'] = group

### PODO significant DEGs

In [ ]:
# Now plot the results
sc.pl.rank_genes_groups(adata_PODO , n_genes=20)

In [ ]:
p = 0.05
log2fc = 0 

deg_podo = sc.get.rank_genes_groups_df(adata_PODO, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_podo.loc[(deg_podo.logfoldchanges > 0) & (deg_podo.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_podo.loc[(deg_podo.logfoldchanges < 0) & (deg_podo.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_podo_fi = deg_podo[deg_podo['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_podo_sig = deg_podo[(deg_podo['pvals_adj'] < p) & (deg_podo['logfoldchanges'].abs() > log2fc)]

deg_podo_sig_sg20 = deg_podo_sig[deg_podo_sig['group']=='age']
deg_podo_sig_sg24 = deg_podo_sig[deg_podo_sig['group']=='sAct']
deg_podo_sig_sg26 = deg_podo_sig[deg_podo_sig['group']=='DR']
deg_podo_sig_sg28 = deg_podo_sig[deg_podo_sig['group']=='combi']

print(deg_podo.shape)
print(deg_podo_fi.shape)
print(deg_podo_sig.shape)
deg_podo

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_podo_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in PODO")
#plt.savefig(pdf_pages, format='pdf')
plt.show()

####  Heatmap Gene Expression 

In [ ]:
#plt.title('Gene expression of last 50 entries sorted by lfcs')
sc.pl.heatmap(adata_PODO, var_names=deg_podo_sig['names'][-50:], groupby='sample', cmap='Blues_r', dendrogram=True, vmin=0, vmax=4)

#plt.title('Gene expression of first 50 entries sorted by lfcs')
sc.pl.heatmap(adata_PODO, var_names=deg_podo_sig['names'][:50], groupby='sample', cmap='Blues_r', dendrogram=True, vmin=0, vmax=4)

#### Dotplot 

In [ ]:
sc.pl.DotPlot(adata_PODO, use_raw=True, var_names = list(set(deg_podo_sig['names'])), groupby='sample', standard_scale="var", title='PODO DEGs', cmap='BuGn', figsize=(300, 5)).savefig(pdf_dotplots ,format='pdf')

In [ ]:
deg_podo_sig[deg_podo_sig['group']=="age"]

In [ ]:
sc.pl.DotPlot(adata_PODO, use_raw=True, var_names = list(deg_podo_sig[deg_podo_sig['group']=="age"].names), groupby='sample', standard_scale="var", title='PODO DEGs', cmap='BuGn', figsize=(300, 5)).savefig(pdf_dotplots ,format='pdf')

#### Vulcanoplot

In [ ]:
# Define the list of groups you want to create plots for
groups_to_plot = ['age', 'sAct', 'DR', 'combi']

# Define a significance threshold (adjusted P-value < 0.05)
significance_threshold = -np.log10(0.05)

# Create a column to classify genes as upregulated or downregulated
deg_podo['Regulation'] = 'Not Significant'  # Initialize as not significant
for group in groups_to_plot:
    condition = (deg_podo['pvals_adj'] < 0.05) & (deg_podo['group'] == group)
    deg_podo.loc[condition, 'Regulation'] = group

# Define colors for significant genes
colors = {'age': 'blue', 'sAct': 'green', 'DR': 'purple', 'combi': 'orange', 'Not Significant': 'gray'}

# Create a main plot with subplots for each group
fig, axs = plt.subplots(len(groups_to_plot), 1, figsize=(20, 10 * len(groups_to_plot)))

# Loop through each group and create a subplot
for i, group in enumerate(groups_to_plot):
    subset_df = deg_podo[deg_podo['Regulation'] == group]
    ax = axs[i]
    ax.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=colors[group], alpha=0.3, label=group)

    # Annotate all significant genes (adjusted P-value < 0.05) with their gene names
    significant_genes = subset_df[subset_df['pvals_adj'] < 0.05]
  #  for index, row in significant_genes.iterrows():
  #      ax.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)

    ax.set_xlabel('Log2(Fold Change)')
    ax.set_xlabel('Log Fold Change')
    ax.set_ylabel('-log10(P-Value)')
    ax.set_title(f'Volcano Plot for {group}')
    ax.axvline(x=-1 * log2fc, linestyle='--', color='gray')
    ax.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
    ax.legend()

# Adjust spacing between subplots
plt.tight_layout()

# Show the main plot with subplots
plt.show()


In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_podo['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_podo.loc[(deg_podo['logfoldchanges'] > 0) & (deg_podo['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_podo.loc[(deg_podo['logfoldchanges'] < 0) & (deg_podo['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_podo.loc[(deg_podo['pvals_adj'] < 0.05) & (deg_podo['group'] == 'age'), 'Regulation'] = 'age'
deg_podo.loc[(deg_podo['pvals_adj'] < 0.05) & (deg_podo['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_podo.loc[(deg_podo['pvals_adj'] < 0.05) & (deg_podo['group'] == 'DR'), 'Regulation'] = 'DR'
deg_podo.loc[(deg_podo['pvals_adj'] < 0.05) & (deg_podo['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'age': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_podo[deg_podo['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_podo[deg_podo['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for Podo')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

#### Expression ERCC1

In [ ]:
genes_of_interest = [
    "Ercc1"
]

# First, filter the dataframe to include only the genes of interest
filtered_df = deg_podo_sig_sg20[deg_podo_sig_sg20['names'].isin(genes_of_interest)]
print(filtered_df)

# Now, create a bar plot
plt.figure(figsize=(15, 6))
# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors = ['green' if x >= 0 else 'red' for x in filtered_df['logfoldchanges']]
plt.bar(filtered_df['names'], filtered_df['logfoldchanges'], color=colors)
plt.xlabel('Genes')
plt.ylabel('Log-Fold Changes')
plt.title('Log-Fold Changes of Genes of Interest SG20 vs. SG18')
plt.xticks(rotation=90)  # Rotate x-axis labels for readability
plt.tight_layout()
plt.show()

# Visualize expression patterns of these mouse genes, e.g., a heatmap
sc.pl.heatmap(adata_PODO, var_names=genes_of_interest, groupby = 'sample', use_raw = True)
sc.pl.violin(adata_PODO, keys=genes_of_interest, groupby = 'sample', use_raw = True)
sc.pl.dotplot(adata_PODO, var_names=genes_of_interest, groupby = 'sample', use_raw = True)
sc.pl.dotplot(adata_PODO, var_names=genes_of_interest, groupby = 'sample', use_raw = True, standard_scale =  'group')
sc.pl.matrixplot(adata_PODO, genes_of_interest, 'sample', dendrogram=True, cmap='Blues', standard_scale='var', colorbar_title='column scaled\nexpression')

#### Expression of Actin Cytoskeleton

In [ ]:
genes_of_interest = [
    "Abl1", "Acta1", "Acta2", "Actb", "Actc1", "Actg1", "Actg2", "Actl6a", "Actn1", "Actn4", 
    "Actr2", "Actr3", "Akt1", "Anln", "Anxa2", "Arf6", "Baiap2", "Cav1", "Ccdc88a", "Ccn2", 
    "Cd44", "Cdc42", "Cdh1", "Cdh5", "Cfl1", "Cftr", "Ctnnb1", "Cttn", "Cxcl12", "Cxcr4", 
    "Dbn1", "Diaph1", "Dmd", "Egfr", "Enah", "Ezr", "Flna", "Fscn1", "Gsn", "Hif1a", "Hspb1", 
    "Icam1", "Il6", "Ilk", "Iqgap1", "Itgav", "Itgb1", "Itgb2", "Itgb3", "Limk1", "Mapk1", 
    "Mapk14", "Mapk3", "Mmp14", "Mmp2", "Mmp9", "Mrtfa", "Msn", "Mtor", "Mybpc3", "Myh7", 
    "Myh9", "Myo6", "Nck1", "Nfkb1", "Pak1", "Pfn1", "Pten", "Ptk2", "Pxn", 
    "Rac1", "Rap1a", "Rdx", "Rhoa", "Rhob", "Rock1", "Rock2", "Smad3", "Src", "Srf", 
    "Stat3", "Tagln", "Tgfb1", "Tln1", "Tmsb4x", "Tnf", "Tnnc1", "Tnni3", 
    #"Tp53", 
    "Tpm1", "Vasp", "Vcl", "Vegfa", "Vim", "Was", "Wasf2", "Wasl", "Wwtr1", "Yap1"
]

#Filter the genes to include only those present in 'adata.var'
valid_genes_of_interest = [gene for gene in genes_of_interest if gene in adata.var_names]
valid_genes_of_interest

oli_genes = ['Dnm1','Dnm2', 'Dnm3', 'Clta']

In [ ]:
# First, filter the dataframe to include only the genes of interest
filtered_df = deg_podo_sig_sg20[deg_podo_sig_sg20['names'].isin(genes_of_interest)]
print(filtered_df)

# Now, create a bar plot
plt.figure(figsize=(15, 6))
# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors = ['green' if x >= 0 else 'red' for x in filtered_df['logfoldchanges']]
plt.bar(filtered_df['names'], filtered_df['logfoldchanges'], color=colors)
plt.xlabel('Genes')
plt.ylabel('Log-Fold Changes')
plt.title('Log-Fold Changes of Genes of Interest SG20 vs. SG18')
plt.xticks(rotation=90)  # Rotate x-axis labels for readability
plt.tight_layout()
plt.show()

# Visualize expression patterns of these mouse genes, e.g., a heatmap
sc.pl.heatmap(adata_PODO, var_names=genes_of_interest, groupby = 'sample', use_raw = True)
sc.pl.violin(adata_PODO, keys=valid_genes_of_interest, groupby = 'sample', use_raw = True)
sc.pl.dotplot(adata_PODO, var_names=genes_of_interest, groupby = 'sample', use_raw = True)
sc.pl.dotplot(adata_PODO, var_names=genes_of_interest, groupby = 'sample', use_raw = True, standard_scale =  'group')
sc.pl.matrixplot(adata_PODO, genes_of_interest, 'sample', dendrogram=True, cmap='Blues', standard_scale='var', colorbar_title='column scaled\nexpression')


In [ ]:
# First, filter the dataframe to include only the genes of interest
filtered_df = deg_podo_sig_sg20[deg_podo_sig_sg20['names'].isin(oli_genes)]
print(filtered_df)

# Now, create a bar plot
plt.figure(figsize=(5, 5))
# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors = ['green' if x >= 0 else 'red' for x in filtered_df['logfoldchanges']]
plt.bar(filtered_df['names'], filtered_df['logfoldchanges'], color=colors)
plt.xlabel('Genes')
plt.ylabel('Log-Fold Changes')
plt.title('Log-Fold Changes of Genes of Interest SG20 vs. SG18')
plt.xticks(rotation=90)  # Rotate x-axis labels for readability
plt.tight_layout()
plt.show()

sc.pl.heatmap(adata_PODO, var_names=oli_genes, groupby = 'sample')
sc.pl.violin(adata_PODO, keys=oli_genes, groupby = 'sample')
sc.pl.dotplot(adata_PODO, var_names=oli_genes, groupby = 'sample')

### vSMC significant DEGs

In [ ]:
sc.pl.rank_genes_groups(adata_vSMC, n_genes = 20)

In [ ]:
p = 0.05
log2fc = 0 

deg_vsmc = sc.get.rank_genes_groups_df(adata_vSMC, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_vsmc.loc[(deg_vsmc.logfoldchanges > 0) & (deg_vsmc.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_vsmc.loc[(deg_vsmc.logfoldchanges < 0) & (deg_vsmc.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_vsmc_fi = deg_vsmc[deg_vsmc['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_vsmc_sig = deg_vsmc[(deg_vsmc['pvals_adj'] < p) & (deg_vsmc['logfoldchanges'].abs() > log2fc)]

deg_vsmc_sig_sg20 = deg_vsmc_sig[deg_vsmc_sig['group']=='age']
deg_vsmc_sig_sg24 = deg_vsmc_sig[deg_vsmc_sig['group']=='sAct']
deg_vsmc_sig_sg26 = deg_vsmc_sig[deg_vsmc_sig['group']=='DR']
deg_vsmc_sig_sg28 = deg_vsmc_sig[deg_vsmc_sig['group']=='combi']

print(deg_vsmc.shape)
print(deg_vsmc_fi.shape)
print(deg_vsmc_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_vsmc_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in vSMC/MC")
plt.savefig(pdf_pages, format='pdf')
plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_vsmc['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_vsmc.loc[(deg_vsmc['logfoldchanges'] > 0) & (deg_vsmc['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_vsmc.loc[(deg_vsmc['logfoldchanges'] < 0) & (deg_vsmc['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_vsmc.loc[(deg_vsmc['pvals_adj'] < 0.05) & (deg_vsmc['group'] == 'age'), 'Regulation'] = 'age'
deg_vsmc.loc[(deg_vsmc['pvals_adj'] < 0.05) & (deg_vsmc['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_vsmc.loc[(deg_vsmc['pvals_adj'] < 0.05) & (deg_vsmc['group'] == 'DR'), 'Regulation'] = 'DR'
deg_vsmc.loc[(deg_vsmc['pvals_adj'] < 0.05) & (deg_vsmc['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'age': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_vsmc[deg_vsmc['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_vsmc[deg_vsmc['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for vSMC')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

### PEC significant DEGs

In [ ]:
sc.pl.rank_genes_groups(adata_PEC, n_genes = 20)

In [ ]:
p = 0.05
log2fc = 0 

deg_pec = sc.get.rank_genes_groups_df(adata_PEC, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_pec.loc[(deg_pec.logfoldchanges > 0) & (deg_pec.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_pec.loc[(deg_pec.logfoldchanges < 0) & (deg_pec.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_pec_fi = deg_pec[deg_pec['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_pec_sig = deg_pec[(deg_pec['pvals_adj'] < p) & (deg_pec['logfoldchanges'].abs() > log2fc)]

deg_pec_sig_sg20 = deg_pec_sig[deg_pec_sig['group']=='age']
deg_pec_sig_sg24 = deg_pec_sig[deg_pec_sig['group']=='sAct']
deg_pec_sig_sg26 = deg_pec_sig[deg_pec_sig['group']=='DR']
deg_pec_sig_sg28 = deg_pec_sig[deg_pec_sig['group']=='combi']

print(deg_pec.shape)
print(deg_pec_fi.shape)
print(deg_pec_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_pec_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in PEC")
plt.savefig(pdf_pages, format='pdf')
plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_pec['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_pec.loc[(deg_pec['logfoldchanges'] > 0) & (deg_pec['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_pec.loc[(deg_pec['logfoldchanges'] < 0) & (deg_pec['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_pec.loc[(deg_pec['pvals_adj'] < 0.05) & (deg_pec['group'] == 'age'), 'Regulation'] = 'age'
deg_pec.loc[(deg_pec['pvals_adj'] < 0.05) & (deg_pec['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_pec.loc[(deg_pec['pvals_adj'] < 0.05) & (deg_pec['group'] == 'DR'), 'Regulation'] = 'DR'
deg_pec.loc[(deg_pec['pvals_adj'] < 0.05) & (deg_pec['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'age': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_pec[deg_pec['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_pec[deg_pec['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for PEC')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

### EC1 (gEC) significant DEGs

In [ ]:
sc.pl.rank_genes_groups(adata_EC1, n_genes = 20)

In [ ]:
p = 0.05
log2fc = 0 

deg_ec1 = sc.get.rank_genes_groups_df(adata_EC1, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_ec1.loc[(deg_ec1.logfoldchanges > 0) & (deg_ec1.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_ec1.loc[(deg_ec1.logfoldchanges < 0) & (deg_ec1.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_ec1_fi = deg_ec1[deg_ec1['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_ec1_sig = deg_ec1[(deg_ec1['pvals_adj'] < p) & (deg_ec1['logfoldchanges'].abs() > log2fc)]

deg_ec1_sig_sg20 = deg_ec1_sig[deg_ec1_sig['group']=='age']
deg_ec1_sig_sg24 = deg_ec1_sig[deg_ec1_sig['group']=='sAct']
deg_ec1_sig_sg26 = deg_ec1_sig[deg_ec1_sig['group']=='DR']
deg_ec1_sig_sg28 = deg_ec1_sig[deg_ec1_sig['group']=='combi']

print(deg_ec1.shape)
print(deg_ec1_fi.shape)
print(deg_ec1_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_ec1_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in EC1")
plt.savefig(pdf_pages, format='pdf')
plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_ec1['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_ec1.loc[(deg_ec1['logfoldchanges'] > 0) & (deg_ec1['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_ec1.loc[(deg_ec1['logfoldchanges'] < 0) & (deg_ec1['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_ec1.loc[(deg_ec1['pvals_adj'] < 0.05) & (deg_ec1['group'] == 'age'), 'Regulation'] = 'age'
deg_ec1.loc[(deg_ec1['pvals_adj'] < 0.05) & (deg_ec1['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_ec1.loc[(deg_ec1['pvals_adj'] < 0.05) & (deg_ec1['group'] == 'DR'), 'Regulation'] = 'DR'
deg_ec1.loc[(deg_ec1['pvals_adj'] < 0.05) & (deg_ec1['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'age': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_ec1[deg_ec1['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_ec1[deg_ec1['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
 #   plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for EC1(gEC)')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

### PT1 significant DEGs

In [ ]:
sc.pl.rank_genes_groups(adata_PT1, n_genes = 20)

In [ ]:
p = 0.05
log2fc = 0 

deg_pt1 = sc.get.rank_genes_groups_df(adata_PT1, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_pt1.loc[(deg_pt1.logfoldchanges > 0) & (deg_pt1.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_pt1.loc[(deg_pt1.logfoldchanges < 0) & (deg_pt1.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_pt1_fi = deg_pt1[deg_pt1['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_pt1_sig = deg_pt1[(deg_pt1['pvals_adj'] < p) & (deg_pt1['logfoldchanges'].abs() > log2fc)]

deg_pt1_sig_sg20 = deg_pt1_sig[deg_pt1_sig['group']=='age']
deg_pt1_sig_sg24 = deg_pt1_sig[deg_pt1_sig['group']=='sAct']
deg_pt1_sig_sg26 = deg_pt1_sig[deg_pt1_sig['group']=='DR']
deg_pt1_sig_sg28 = deg_pt1_sig[deg_pt1_sig['group']=='combi']

print(deg_pt1.shape)
print(deg_pt1_fi.shape)
print(deg_pt1_sig.shape)


#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_pt1_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in PT1")
plt.savefig(pdf_pages, format='pdf')
plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_pt1['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_pt1.loc[(deg_pt1['logfoldchanges'] > 0) & (deg_pt1['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_pt1.loc[(deg_pt1['logfoldchanges'] < 0) & (deg_pt1['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_pt1.loc[(deg_pt1['pvals_adj'] < 0.05) & (deg_pt1['group'] == 'age'), 'Regulation'] = 'age'
deg_pt1.loc[(deg_pt1['pvals_adj'] < 0.05) & (deg_pt1['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_pt1.loc[(deg_pt1['pvals_adj'] < 0.05) & (deg_pt1['group'] == 'DR'), 'Regulation'] = 'DR'
deg_pt1.loc[(deg_pt1['pvals_adj'] < 0.05) & (deg_pt1['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'age': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_pt1[deg_pt1['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_pt1[deg_pt1['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for PT1')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

#### Dotplot 

### PT2 significant DEGs

In [ ]:
p = 0.05
log2fc = 0 

deg_pt2 = sc.get.rank_genes_groups_df(adata_PT2, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_pt2.loc[(deg_pt2.logfoldchanges > 0) & (deg_pt2.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_pt2.loc[(deg_pt2.logfoldchanges < 0) & (deg_pt2.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_pt2_fi = deg_pt2[deg_pt2['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_pt2_sig = deg_pt2[(deg_pt2['pvals_adj'] < p) & (deg_pt2['logfoldchanges'].abs() > log2fc)]

deg_pt2_sig_sg20 = deg_pt2_sig[deg_pt2_sig['group']=='age']
deg_pt2_sig_sg24 = deg_pt2_sig[deg_pt2_sig['group']=='sAct']
deg_pt2_sig_sg26 = deg_pt2_sig[deg_pt2_sig['group']=='DR']
deg_pt2_sig_sg28 = deg_pt2_sig[deg_pt2_sig['group']=='combi']

print(deg_pt2.shape)
print(deg_pt2_fi.shape)
print(deg_pt2_sig.shape)


#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_pt2_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in PT2")
plt.savefig(pdf_pages, format='pdf')
plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_pt2['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_pt2.loc[(deg_pt2['logfoldchanges'] > 0) & (deg_pt2['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_pt2.loc[(deg_pt2['logfoldchanges'] < 0) & (deg_pt2['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_pt2.loc[(deg_pt2['pvals_adj'] < 0.05) & (deg_pt2['group'] == 'age'), 'Regulation'] = 'age'
deg_pt2.loc[(deg_pt2['pvals_adj'] < 0.05) & (deg_pt2['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_pt2.loc[(deg_pt2['pvals_adj'] < 0.05) & (deg_pt2['group'] == 'DR'), 'Regulation'] = 'DR'
deg_pt2.loc[(deg_pt2['pvals_adj'] < 0.05) & (deg_pt2['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'age': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_pt2[deg_pt2['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_pt2[deg_pt2['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for PT-2')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

#### Dotplot

In [ ]:
sc.pl.DotPlot(adata_PT2, use_raw=True, var_names = list(set(deg_pt2_sig['names'])), groupby='sample', standard_scale="var", title='PT2 DEGs', cmap='BuGn', figsize=(300, 5)).savefig(pdf_dotplots ,format='pdf')

### EC2 significant DEGs

In [ ]:
p = 0.05
log2fc = 0

deg_ec2 = sc.get.rank_genes_groups_df(adata_EC2, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_ec2.loc[(deg_ec2.logfoldchanges > 0) & (deg_ec2.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_ec2.loc[(deg_ec2.logfoldchanges < 0) & (deg_ec2.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_ec2_fi = deg_ec2[deg_ec2['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_ec2_sig = deg_ec2[(deg_ec2['pvals_adj'] < p) & (deg_ec2['logfoldchanges'].abs() > log2fc)]

deg_ec2_sig_sg20 = deg_ec2_sig[deg_ec2_sig['group']=='age']
deg_ec2_sig_sg24 = deg_ec2_sig[deg_ec2_sig['group']=='sAct']
deg_ec2_sig_sg26 = deg_ec2_sig[deg_ec2_sig['group']=='DR']
deg_ec2_sig_sg28 = deg_ec2_sig[deg_ec2_sig['group']=='combi']

print(deg_ec2.shape)
print(deg_ec2_fi.shape)
print(deg_ec2_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_ec2_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,4))
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in EC2")
plt.show()

####  Heatmap Gene Expression 

#### Dotplot 

### TAL1 significant DEGs

In [ ]:
p = 0.05
log2fc = 0 

deg_tal1 = sc.get.rank_genes_groups_df(adata_TAL1, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_tal1.loc[(deg_tal1.logfoldchanges > 0) & (deg_tal1.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_tal1.loc[(deg_tal1.logfoldchanges < 0) & (deg_tal1.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_tal1_fi = deg_tal1[deg_tal1['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_tal1_sig = deg_tal1[(deg_tal1['pvals_adj'] < p) & (deg_tal1['logfoldchanges'].abs() > log2fc)]

deg_tal1_sig_sg20 = deg_tal1_sig[deg_tal1_sig['group']=='age']
deg_tal1_sig_sg24 = deg_tal1_sig[deg_tal1_sig['group']=='sAct']
deg_tal1_sig_sg26 = deg_tal1_sig[deg_tal1_sig['group']=='DR']
deg_tal1_sig_sg28 = deg_tal1_sig[deg_tal1_sig['group']=='combi']

print(deg_tal1.shape)
print(deg_tal1_fi.shape)
print(deg_tal1_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_tal1_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,4))
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in TAL1")
plt.show()

####  Heatmap Gene Expression 

#### Dotplot 

### IMM significant DEGs

In [ ]:
p = 0.05
log2fc = 0 

deg_imm = sc.get.rank_genes_groups_df(adata_IMM, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_imm.loc[(deg_imm.logfoldchanges > 0) & (deg_imm.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_imm.loc[(deg_imm.logfoldchanges < 0) & (deg_imm.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_imm_fi = deg_imm[deg_imm['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_imm_sig = deg_imm[(deg_imm['pvals_adj'] < p) & (deg_imm['logfoldchanges'].abs() > log2fc)]

deg_imm_sig_sg20 = deg_imm_sig[deg_imm_sig['group']=='age']
deg_imm_sig_sg24 = deg_imm_sig[deg_imm_sig['group']=='sAct']
deg_imm_sig_sg26 = deg_imm_sig[deg_imm_sig['group']=='DR']
deg_imm_sig_sg28 = deg_imm_sig[deg_imm_sig['group']=='combi']

print(deg_imm.shape)
print(deg_imm_fi.shape)
print(deg_imm_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_imm_sig.pivot(index='group', columns='names', values='logfoldchanges')
print(heatmap_matrix)
#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in IMM")
plt.savefig(pdf_pages, format='pdf')
plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_imm['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_imm.loc[(deg_imm['logfoldchanges'] > 0) & (deg_imm['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_imm.loc[(deg_imm['logfoldchanges'] < 0) & (deg_imm['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_imm.loc[(deg_imm['pvals_adj'] < 0.05) & (deg_imm['group'] == 'age'), 'Regulation'] = 'age'
deg_imm.loc[(deg_imm['pvals_adj'] < 0.05) & (deg_imm['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_imm.loc[(deg_imm['pvals_adj'] < 0.05) & (deg_imm['group'] == 'DR'), 'Regulation'] = 'DR'
deg_imm.loc[(deg_imm['pvals_adj'] < 0.05) & (deg_imm['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'age': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_imm[deg_imm['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_imm[deg_imm['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for IMM')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

#### Dotplot 

In [ ]:
# filter upregulated
deg_imm_sg20up = deg_imm_sig[(deg_imm_sig['logfoldchanges'] > 0) & (deg_imm_sig['group'] == 'sAct')]

sc.pl.DotPlot(adata_IMM, use_raw=True, var_names =deg_imm_sg20up.names, groupby='sample', standard_scale="var", title='IMM DEGs', cmap='BuGn', figsize=(300, 5)).savefig(pdf_dotplots ,format='pdf')

In [ ]:
sc.pl.DotPlot(adata_IMM, use_raw=True, var_names = list(set(deg_imm_sig['names'])), groupby='sample', standard_scale="var", title='IMM DEGs', cmap='BuGn', figsize=(300, 5)).savefig(pdf_dotplots ,format='pdf')

# Check Qpct, Qpctl, Ccl2

## pseudobulk

In [ ]:
p = 0.05

In [ ]:
# deg is a pandas DataFrame with a column called 'names'
filtered_genes = deg[deg['names'].isin(['Qpct', 'Qpctl', 'Ccl2'])]
# Display the filtered results
filtered_genes = filtered_genes.drop(['scores', 'pvals'], axis=1)
filtered_genes.to_csv('/data/projects/manuela/aging/scRNA_aging-mouse/tables/specific_analysis_results/glutaminyl cyclases_kidney_sn_pseudobulk.csv', index=False)

filtered_genes#[filtered_genes['pvals_adj'] < p]

## podocytes

In [ ]:
# deg_podo is a pandas DataFrame with a column called 'names'
filtered_genes_podo = deg_podo[deg_podo['names'].isin(['Qpct', 'Qpctl', 'Ccl2'])]
# Display the filtered results
filtered_genes_podo = filtered_genes_podo.drop(['scores', 'pvals', 'sign'], axis=1)
filtered_genes_podo.to_csv('/data/projects/manuela/aging/scRNA_aging-mouse/tables/specific_analysis_results/glutaminyl cyclases_kidney_sn_podo.csv', index=False)
filtered_genes_podo

## immune cells

In [ ]:
# deg_imm is a pandas DataFrame with a column called 'names'
filtered_genes_imm = deg_imm[deg_imm['names'].isin(['Qpct', 'Qpctl', 'Ccl2'])]

filtered_genes_imm = filtered_genes_imm.drop(['scores', 'pvals', 'sign'], axis=1)
filtered_genes_imm.to_csv('/data/projects/manuela/aging/scRNA_aging-mouse/tables/specific_analysis_results/glutaminyl cyclases_kidney_sn_immune.csv', index=False)
# Display the filtered results
filtered_genes_imm

# Write DEG csv files

In [ ]:
# Write Age vs Ctrl specifically

deg_sig_up_sg20.to_csv(output_dir + 'up_all_sig_AgevsCtrl.csv', index = False)
deg_sig_dwn_sg20.to_csv(output_dir + 'dwn_all_sig_AgevsCtrl.csv', index = False)


deg_podo_sig_sg20.to_csv(output_dir + 'deg_podo_sig_AgevsCtrl.csv', index = False)
deg_vsmc_sig_sg20.to_csv(output_dir + 'deg_vsmc_sig_AgevsCtrl.csv', index = False)
deg_ec1_sig_sg20.to_csv(output_dir + 'deg_ec1_sig_AgevsCtrl.csv', index = False)
#deg_ec1.to_csv(output_dir + 'deg_ec1.csv', index = False) 
deg_pec_sig_sg20.to_csv(output_dir + 'deg_pec_sig_AgevsCtrl.csv', index = False)

deg_imm_sig_sg20.to_csv(output_dir + 'deg_imm_sig_AgevsCtrl.csv', index = False)
deg_pt1_sig_sg20.to_csv(output_dir + 'deg_pt1_sig_AgevsCtrl.csv', index = False)
deg_pt2_sig_sg20.to_csv(output_dir + 'deg_pt2_sig_AgevsCtrl.csv', index = False)

In [ ]:
deg_sig.to_csv(output_dir + 'deg_sig.csv', index = False)

deg_podo_sig.to_csv(output_dir + 'deg_podo_sig.csv', index = False)
deg_vsmc_sig.to_csv(output_dir + 'deg_vsmc_sig.csv', index = False)
deg_ec1_sig.to_csv(output_dir + 'deg_ec1_sig.csv', index = False)
deg_pec_sig.to_csv(output_dir + 'deg_pec_sig.csv', index = False)

deg_imm_sig.to_csv(output_dir + 'deg_imm_sig.csv', index = False)
deg_pt1_sig.to_csv(output_dir + 'deg_pt1_sig.csv', index = False)
deg_pt2_sig.to_csv(output_dir + 'deg_pt2_sig.csv', index = False)

# Enrichment: Over-representation analysis (Enrichr API)

In [ ]:
from gseapy.scipalette import SciPalette
NbDr = SciPalette.create_colormap() 

## Pseudobulk

### Tabula Muris

In [ ]:
gene_sets = gp.read_gmt('/data/projects/manuela/aging_muscle/msigdb/m8.all.v2023.2.Mm.symbols.gmt')
# Filter the gene sets to only include those starting with 'TABULA_MURIS_SENIS'
gene_sets = {k: v for k, v in gene_sets.items() if k.startswith('TABULA_MURIS_SENIS')}
gene_sets = {k.split('_', maxsplit=3)[-1]: v for k, v in gene_sets.items()}
#

In [ ]:
#print(list(gene_sets.keys()))

all_genes = [gene for genes in gene_sets.values() for gene in genes]
unique_genes = list(set(all_genes))
unique_genes

In [ ]:
# Upregulated genes

res_m8 = gp.enrichr(gene_list=deg_sig['names'].tolist(),
                       gene_sets=gene_sets,
                       organism='mouse')

res_m8_up = gp.enrichr(gene_list=deg_sig_up['names'].tolist(),
                       gene_sets=gene_sets,
                       organism='mouse')

# Downregulated genes
res_m8_dwn = gp.enrichr(gene_list=deg_sig_dwn['names'].tolist(),
                        gene_sets=gene_sets,
                        organism='mouse'
                        )

In [ ]:
# Example of creating a heatmap
sns.heatmap(res_m8_up.res2d.set_index('Term')[['Adjusted P-value']], annot=True, cmap='viridis')
plt.title('Heatmap of GSEA Results')
plt.show()

In [ ]:
gp.dotplot(res_m8.res2d, figsize=(8, 10), top_term=50, title=f" Aging enrichment - Tabula muris senis", cmap=plt.cm.autumn_r)

gp.dotplot(res_m8_up.res2d, figsize=(8, 10), top_term=20, title=f" Up in Aging - Tabula muris senis", cmap=plt.cm.autumn_r)
gp.dotplot(res_m8_dwn.res2d, figsize=(8, 10), top_term=20, title=f" Down in Aging enrichment - Tabula muris senis", cmap=plt.cm.winter_r)

## PODO

In [ ]:
# List of dataframe names
podo_names = ['deg_podo_sig_sg20', 'deg_podo_sig_sg24', 'deg_podo_sig_sg26', 'deg_podo_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in podo_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_PODO, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(50, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_PODO, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(50, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

    enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        #gsplot.dotplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
        dotplot(enr_full.results, top_term=20, title=f"{df_short_name} KEGG all", cmap='RdBu_r', cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'])
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in podo_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    print(degs_up.names)
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

    enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]

    # dotplot
    gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r)
    plt.show()

    gp.dotplot(enr_dw.res2d,
               figsize=(3,5),
               title=f"{df_short_name} GO BP downregulated",
               cmap = plt.cm.winter_r,
               size=5)
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'])
    plt.show()

### SG20

In [ ]:
glist_sg20 = deg_podo_sig_sg20['names'].squeeze().str.strip().tolist()
#print(glist_sg20)

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_podo_sg20.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_podo_sg20.csv')

# Visualize the results
#fig = gsplot.plt.figure()
gsplot.barplot(enrichr_results.res2d, title='SG20 KEGG Enrichment analysis PODO')
#fig.savefig(test_pdf, format='pdf')

dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG20 KEGG Enrichment analysis PODO', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG20 GO BP Enrichment analysis PODO')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG20 GO BP Enrichment analysis PODO', cmap='RdBu_r')

In [ ]:
#test_pdf.close()

### SG24

In [ ]:
glist_sg24 = deg_podo_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_podo_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_podo_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis PODO')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis PODO', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis PODO')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis PODO', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_podo_sig_sg26) 

glist_sg26 = deg_podo_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,organism='Mouse',gene_sets='KEGG_2019_Mouse', cutoff=0.05)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )
enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_podo_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_podo_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis PODO')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis PODO', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis PODO')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis PODO', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_podo_sig_sg28) 

glist_sg28 = deg_podo_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_podo_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_podo_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis PODO')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis PODO', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis PODO')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis PODO', cmap='RdBu_r')

## vSMC

In [ ]:
# List of dataframe names
vsmc_names = ['deg_vsmc_sig_sg20', 'deg_vsmc_sig_sg24', 'deg_vsmc_sig_sg26', 'deg_vsmc_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in vsmc_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    sc.pl.DotPlot(adata_vSMC, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(17, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_vSMC, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_vSMC, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in vsmc_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
  
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

    enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
    # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3,5), title=f"{df_short_name} GO BP downregulated", cmap = plt.cm.winter_r)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all")
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### SG20

In [ ]:
glist_sg20 = deg_vsmc_sig_sg20['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_vsmc_sg20.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_vsmc_sg20.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG20 KEGG Enrichment analysis vSMC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG20 KEGG Enrichment analysis vSMC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG20 GO BP Enrichment analysis vSMC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG20 GO BP Enrichment analysis vSMC', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_vsmc_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )
enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_vsmc_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_vsmc_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis vSMC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis vSMC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis vSMC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis vSMC', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_vsmc_sig_sg26) 

glist_sg26 = deg_vsmc_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,organism='Mouse',gene_sets='KEGG_2019_Mouse',cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_vsmc_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_vsmc_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis vSMC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis vSMC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis vSMC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis vSMC', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_vsmc_sig_sg28) 

glist_sg28 = deg_vsmc_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_vsmc_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_vsmc_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis vSMC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis vSMC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis vSMC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis vSMC', cmap='RdBu_r')

## PEC

In [ ]:
# List of dataframe names
pec_names = ['deg_pec_sig_sg20', 'deg_pec_sig_sg24', 'deg_pec_sig_sg26', 'deg_pec_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in pec_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    sc.pl.DotPlot(adata_PEC, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(17, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_PEC, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_PEC, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in pec_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
  
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

    enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
    # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3,5), title=f"{df_short_name} GO BP downregulated", cmap = plt.cm.winter_r)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all")
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### SG20

In [ ]:
glist_sg20 = deg_pec_sig_sg20['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pec_sg20.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pec_sg20.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG20 KEGG Enrichment analysis PEC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG20 KEGG Enrichment analysis PEC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG20 GO BP Enrichment analysis PEC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG20 GO BP Enrichment analysis PEC', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_pec_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pec_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pec_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis PEC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis PEC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis PEC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis PEC', cmap='RdBu_r')

### SG26

In [ ]:
glist_sg26 = deg_pec_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,organism='Mouse',gene_sets='KEGG_2019_Mouse',cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )
enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pec_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pec_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis PEC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis PEC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis PEC', cutoff = 0.5)
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis PEC', cmap='RdBu_r')

### SG28

In [ ]:
glist_sg28 = deg_pec_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pec_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pec_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis PEC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis PEC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis PEC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis PEC', cmap='RdBu_r')

## EC1

In [ ]:
# List of dataframe names
ec1_names = ['deg_ec1_sig_sg20', 'deg_ec1_sig_sg24', 'deg_ec1_sig_sg26', 'deg_ec1_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in ec1_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    sc.pl.DotPlot(adata_EC1, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_EC1, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_EC1, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in ec1_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                       gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()
    
    
    

### SG20

In [ ]:
glist_sg20 = deg_ec1_sig_sg20['names'].squeeze().str.strip().tolist()
print(deg_ec1_sig_sg20)
#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse')

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_ec1_sg20.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_ec1_sg20.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG20 KEGG Enrichment analysis EC1', cutoff = 0.5)
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG20 KEGG Enrichment analysis EC1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG20 GO BP Enrichment analysis EC1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG20 GO BP Enrichment analysis EC1', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_ec1_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse')

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_ec1_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_ec1_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis EC1', cutoff = 0.5)
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis EC1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis EC1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis EC1', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_ec1_sig_sg26) 

glist_sg26 = deg_ec1_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,organism='Mouse',gene_sets='KEGG_2019_Mouse')

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_ec1_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_ec1_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis EC1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis EC1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis EC1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis EC1', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_ec1_sig_sg28) 

glist_sg28 = deg_ec1_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse')

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_ec1_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_ec1_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis EC1', cutoff = 0.5)
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis EC1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis EC1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis EC1', cmap='RdBu_r')

## PT1

In [ ]:
# List of dataframe names
pt1_names = ['deg_pt1_sig_sg20', 'deg_pt1_sig_sg24', 'deg_pt1_sig_sg26', 'deg_pt1_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in pt1_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
  #  sc.pl.DotPlot(adata_PT1, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    #sc.pl.DotPlot(adata_PT1, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
   # sc.pl.DotPlot(adata_PT1, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in pt1_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                       gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()
    
    
    

### SG20

In [ ]:
glist_sg20 = deg_pt1_sig_sg20['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.05)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pt1_sg20.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pt1_sg20.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG20 KEGG Enrichment analysis PT1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG20 KEGG Enrichment analysis PT1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG20 GO BP Enrichment analysis PT1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG20 GO BP Enrichment analysis PT1', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_pt1_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pt1_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pt1_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis PT1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis PT1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis PT1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis PT1', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_pt1_sig_sg26) 

glist_sg26 = deg_pt1_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse')


enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pt1_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pt1_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis PT1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis PT1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis PT1', cutoff = 0.5)
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis PT1', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_pt1_sig_sg28) 

glist_sg28 = deg_pt1_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,organism='Mouse',gene_sets='KEGG_2019_Mouse')


enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pt1_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pt1_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis PT1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis PT1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis PT1')
dotplot(enrichr_results_gobp.results, cutoff=0.8, top_term=20, title='SG28 GO BP Enrichment analysis PT1', cmap='RdBu_r')

## PT2

In [ ]:
# List of dataframe names
pt2_names = ['deg_pt2_sig_sg20', 'deg_pt2_sig_sg24', 'deg_pt2_sig_sg26', 'deg_pt2_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in pt2_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
   # sc.pl.DotPlot(adata_PT2, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
   # sc.pl.DotPlot(adata_PT2, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
  #  sc.pl.DotPlot(adata_PT2, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in pt2_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                       gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()
    
    
    

### SG20

In [ ]:
glist_sg20 = deg_pt2_sig_sg20['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.05)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pt2_sg20.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pt2_sg20.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG20 KEGG Enrichment analysis PT2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG20 KEGG Enrichment analysis PT2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG20 GO BP Enrichment analysis PT2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG20 GO BP Enrichment analysis PT2', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_pt2_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pt2_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pt2_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis PT2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis PT2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis PT2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis PT2', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_pt2_sig_sg26) 

glist_sg26 = deg_pt2_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse')


enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pt2_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pt2_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis PT2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis PT2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis PT2', cutoff = 0.5)
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis PT2', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_pt2_sig_sg28) 

glist_sg28 = deg_pt2_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,organism='Mouse',gene_sets='KEGG_2019_Mouse')


enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_pt2_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_pt2_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis PT2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis PT2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis PT2')
dotplot(enrichr_results_gobp.results, cutoff=0.8, top_term=20, title='SG28 GO BP Enrichment analysis PT2', cmap='RdBu_r')

## EC2

### SG20

In [ ]:
glist_sg20 = deg_ec2_sig_sg20['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_ec2_sg20.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_ec2_sg20.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG20 KEGG Enrichment analysis EC2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG20 KEGG Enrichment analysis EC2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG20 GO BP Enrichment analysis EC2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG20 GO BP Enrichment analysis EC2', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_ec2_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_ec2_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_ec2_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis EC2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis EC2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis EC2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis EC2', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_ec2_sig_sg26) 

glist_sg26 = deg_ec2_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_ec2_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_ec2_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis EC2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis EC2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis EC2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis EC2', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_ec2_sig_sg28) 

glist_sg28 = deg_ec2_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_ec2_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_ec2_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis EC2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis EC2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis EC2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis EC2', cmap='RdBu_r')

## TAL1

### SG20

In [ ]:
glist_sg20 = deg_tal1_sig_sg20['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
#enrichr_results = gp.enrichr(gene_list= glist_sg20, organism='Mouse', gene_sets='KEGG_2019_Mouse'# background='mmusculus_gene_ensembl')

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

#enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_tal1_sg20.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_tal1_sg20.csv')

# Visualize the results
#gsplot.barplot(enrichr_results.res2d, title='SG20 Enrichment Analysis Results')
#dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG20 Enrichment analysis TAL1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG20 GO BP Enrichment analysis TAL1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG20 GO BP Enrichment analysis TAL1', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_tal1_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_tal1_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_tal1_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis TAL1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis TAL1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis TAL1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis TAL1', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_tal1_sig_sg26) 

glist_sg26 = deg_tal1_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_tal1_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_tal1_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis TAL1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis TAL1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis TAL1', cmap='RdBu_r')

### SG28

In [ ]:
glist_sg28 = deg_tal1_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_tal1_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_tal1_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis TAL1', cmap='RdBu_r')

#gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis TAL1')
#dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis TAL1', cmap='RdBu_r')

## IMM

In [ ]:
# List of dataframe names
imm_names = ['deg_imm_sig_sg20', 'deg_imm_sig_sg24', 'deg_imm_sig_sg26', 'deg_imm_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in imm_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
   # sc.pl.DotPlot(adata_IMM, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
   # sc.pl.DotPlot(adata_IMM, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
   # sc.pl.DotPlot(adata_IMM, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(10, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in imm_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                       gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} GO BP all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()
    
    
    

### SG20

In [ ]:
glist_sg20 = deg_imm_sig_sg20['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg20,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_imm_sg20.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_imm_sg20.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG20 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG20 KEGG Enrichment analysis IMM', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG20 GO BP Enrichment analysis IMM')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG20 GO BP Enrichment analysis IMM', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_imm_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_imm_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_imm_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis IMM', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis IMM')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis IMM', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_imm_sig_sg26) 

glist_sg26 = deg_imm_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_imm_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_imm_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis IMM', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis IMM')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis IMM', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_imm_sig_sg28) 

glist_sg28 = deg_imm_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/kegg_imm_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG18/gobp_imm_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis IMM', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis IMM')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis IMM', cmap='RdBu_r')

In [ ]:
#sc.pl.umap(adata, color='celltype')

## Pathway Activity Inference

### Either PROGENy or DoRothEA model

PROGENy is a comprehensive resource containing a curated collection of pathways and their target genes, with weights for each interaction

In [ ]:
#model = dc.get_progeny(organism = 'Mus musculus')
#model

#model = dc.get_progeny(organism='human', top=100)
#model

import omnipath
from pypath.utils import homology, mapping

progeny = omnipath.requests.Annotations.get(resources = 'PROGENy', wide = True)

#progeny['mouse_uniprot'] = [homology.translate(u, 10090) for u in progeny.uniprot]
#progeny = progeny.explode('mouse_uniprot')
#progeny['mouse_genesymbol'] = [mapping.label(u, ncbi_tax_id = 10090) for u in progeny.mouse_uniprot]

In [ ]:
model = dc.get_dorothea(organism = 'mouse')
model

### Activity inference with Multivariate Linear Model

In [ ]:
dc.run_mlm(mat=adata, net=model, source='source', target='target', weight='weight', verbose=True, min_n=1)

In [ ]:
adata.obs.columns

In [ ]:
acts = dc.get_acts(adata, obsm_key='mlm_estimate')
acts

In [ ]:
sc.pl.umap(acts)
#sc.pl.umap(acts, color='Trail', vcenter=0, cmap='coolwarm')

In [ ]:
mean_acts = dc.summarize_acts(acts, groupby='celltype', min_std=0)
mean_acts

In [ ]:
sns.clustermap(mean_acts, xticklabels=mean_acts.columns, vmin=-2, vmax=2, cmap='coolwarm', row_cluster=False, figsize=(30, 10))
plt.show()

## MSigDB gene sets

In [ ]:
#dc.show_resources()

In [ ]:
msigdb_mouse = msigdb.msigdb_download_collections(organism = 'mouse')
msigdb_mouse = pd.DataFrame(
    [
        (collname, collcode, gset, gene)
        for (collname, collcode), coll in msigdb_mouse.items()
        for gset, genes in coll.items()
        for gene in genes
    ],
    columns = ['collection', 'code', 'geneset', 'genesymbol']
)

msigdb_mouse

In [ ]:
dc.run_ora(mat=adata, net=msigdb, source='geneset', target='genesymbol', verbose=True)

In [ ]:
acts = dc.get_acts(adata, obsm_key='ora_estimate')
acts

In [ ]:
adata.obsm['ora_estimate']

In [ ]:
sc.pl.umap(acts, color='REACTOME_INTRINSIC_PATHWAY_OF_FIBRIN_CLOT_FORMATION')

In [ ]:
mean_enr = dc.summarize_acts(acts, groupby='celltype', min_std=0)
mean_enr

In [ ]:
sns.clustermap(mean_enr, xticklabels=mean_enr.columns, vmax=20, cmap='viridis')
plt.show()

## Enrichment with Over Representation Analysis

In [ ]:
dc.run_ora(mat=adata, net=msigdb, source='geneset', target='genesymbol', verbose=True)

In [ ]:
acts = dc.get_acts(adata, obsm_key='ora_estimate')

In [ ]:
absolute_sums = np.abs(adata.obsm['ora_estimate']).sum(axis=0)
sorted_indices = np.argsort(absolute_sums)[::-1]  # Sort indices in descending order
sorted_indices = np.argsort(absolute_sums)[::-1]  # Sort indices in descending order

top_n = 10  
top_pathways = acts[:, sorted_indices[:top_n]]
sorted_indices.index

In [ ]:
for i, pathway in enumerate(sorted_indices.index):
    sc.pl.umap(acts, color=pathway)

In [ ]:
for i, pathway in enumerate(top_pathways.columns):
    adata.obs[pathway] = top_pathways[:, i]


In [ ]:
# Assuming `top_pathways.columns` contains the names of the top pathways
sc.pl.umap(adata, color=top_pathways.columns, legend_loc='on data')


# Close pdfs

In [ ]:
pdf_pages.close()
pdf_dotplots.close()